<a href="https://colab.research.google.com/github/suryanshshah2006/ZAN-Multimodal-Neuroimaging/blob/main/zan_fmri_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os, gc, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

import tensorflow as tf
from tensorflow.keras import layers, Model

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

FEATURES_DIR = '/content/drive/MyDrive/ZAN_Research/features_v2'
print('Setup complete ✅')

Setup complete ✅


In [3]:
class OrthogonalAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads, dropout=0.4):
        super().__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim  = embed_dim // num_heads
        self.q   = layers.Dense(embed_dim)
        self.k   = layers.Dense(embed_dim)
        self.v   = layers.Dense(embed_dim)
        self.out = layers.Dense(embed_dim)
        self.dropout = layers.Dropout(dropout)

    def call(self, x, training=False):
        B = tf.shape(x)[0]
        N = tf.shape(x)[1]
        Q = tf.transpose(tf.reshape(self.q(x), (B, N, self.num_heads, self.head_dim)), (0,2,1,3))
        K = tf.transpose(tf.reshape(self.k(x), (B, N, self.num_heads, self.head_dim)), (0,2,1,3))
        V = tf.transpose(tf.reshape(self.v(x), (B, N, self.num_heads, self.head_dim)), (0,2,1,3))
        scale = tf.math.sqrt(tf.cast(self.head_dim, tf.float32))
        attn  = tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) / scale, axis=-1)
        self.attention_weights = self.dropout(attn, training=training)
        out = tf.reshape(
            tf.transpose(tf.matmul(self.attention_weights, V), (0,2,1,3)),
            (B, N, self.embed_dim)
        )
        return self.out(out)


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.4):
        super().__init__()
        self.attention = OrthogonalAttention(embed_dim, num_heads, dropout)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(embed_dim)
        ])
        self.norm1   = layers.LayerNormalization(epsilon=1e-6)
        self.norm2   = layers.LayerNormalization(epsilon=1e-6)
        self.dropout = layers.Dropout(dropout)

    def call(self, x, training=False):
        x = self.norm1(x + self.dropout(self.attention(x, training=training), training=training))
        x = self.norm2(x + self.dropout(self.ffn(x, training=training),       training=training))
        return x


class BrainNetTransformer(Model):
    def __init__(self, num_rois=60, embed_dim=64, num_heads=4,
                 ff_dim=128, num_layers=2, dropout=0.4):
        super().__init__()
        self.embedding = layers.Dense(embed_dim)
        self.cls_token = tf.Variable(tf.zeros((1, 1, embed_dim)), trainable=True)
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ]
        self.norm       = layers.LayerNormalization(epsilon=1e-6)
        self.dropout    = layers.Dropout(dropout)
        self.classifier = tf.keras.Sequential([
            layers.Dense(32, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(1, activation='sigmoid')
        ])

    def call(self, x, training=False):
        B   = tf.shape(x)[0]
        x   = self.embedding(x)
        cls = tf.tile(self.cls_token, (B, 1, 1))
        x   = tf.concat([cls, x], axis=1)
        for block in self.transformer_blocks:
            x = block(x, training=training)
        cls_out = self.dropout(self.norm(x[:, 0, :]), training=training)
        return self.classifier(cls_out)


def run_transformer_fold(X_tr, y_tr, X_te, y_te, num_rois=60, num_layers=2):
    model = BrainNetTransformer(
        num_rois=num_rois, embed_dim=64, num_heads=4,
        ff_dim=128, num_layers=num_layers, dropout=0.5
    )
    class_weight = {0: 1.0, 1: float(np.sum(y_tr==0)) / max(np.sum(y_tr==1), 1)}
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy')
    cb = [tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)]
    model.fit(X_tr, y_tr, validation_data=(X_te, y_te),
              epochs=100, batch_size=8, callbacks=cb,
              class_weight=class_weight, verbose=0)
    probs = model.predict(X_te, verbose=0).flatten()
    tf.keras.backend.clear_session()
    gc.collect()
    return probs


def evaluate_model(y_true, y_pred_proba):
    y_pred = (y_pred_proba > 0.5).astype(int)
    return {
        'acc': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_pred_proba),
        'f1' : f1_score(y_true, y_pred, zero_division=0)
    }

print('Architecture + helpers defined ✅')

Architecture + helpers defined ✅


In [4]:
def run_full_cv(X_flat, X_3d, y, num_rois, dataset_name, n_splits=10, seed=42):
    """
    Runs 10-fold CV for SVM, MLP, Transformer, Ensemble.
    Returns a list of per-fold dicts (one row per fold per model).
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_records = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_flat, y)):
        print(f'  [{dataset_name}] Fold {fold+1}/{n_splits}', end='\r')
        y_tr, y_te = y[tr_idx], y[te_idx]

        # --- preprocessing (identical to original) ---
        scaler  = StandardScaler()
        Xtr_sc  = scaler.fit_transform(X_flat[tr_idx])
        Xte_sc  = scaler.transform(X_flat[te_idx])

        sel     = SelectKBest(f_classif, k=300)
        Xtr_sel = sel.fit_transform(Xtr_sc, y_tr)
        Xte_sel = sel.transform(Xte_sc)

        # --- SVM ---
        svm = SVC(kernel='rbf', probability=True, random_state=seed)
        svm.fit(Xtr_sel, y_tr)
        p_svm = svm.predict_proba(Xte_sel)[:, 1]
        m_svm = evaluate_model(y_te, p_svm)
        fold_records.append({'dataset': dataset_name, 'model': 'SVM-RBF', 'fold': fold+1,
                              'accuracy': m_svm['acc'], 'auc': m_svm['auc'], 'f1': m_svm['f1']})

        # --- MLP ---
        mlp = MLPClassifier(hidden_layer_sizes=(128,64), alpha=0.1, max_iter=500, random_state=seed)
        mlp.fit(Xtr_sel, y_tr)
        p_mlp = mlp.predict_proba(Xte_sel)[:, 1]
        m_mlp = evaluate_model(y_te, p_mlp)
        fold_records.append({'dataset': dataset_name, 'model': 'MLP', 'fold': fold+1,
                              'accuracy': m_mlp['acc'], 'auc': m_mlp['auc'], 'f1': m_mlp['f1']})

        # --- BrainNetTransformer ---
        p_tr = run_transformer_fold(X_3d[tr_idx], y_tr, X_3d[te_idx], y_te, num_rois=num_rois)
        m_tr = evaluate_model(y_te, p_tr)
        fold_records.append({'dataset': dataset_name, 'model': 'BrainNetTransformer', 'fold': fold+1,
                              'accuracy': m_tr['acc'], 'auc': m_tr['auc'], 'f1': m_tr['f1']})

        # --- Ensemble (SVM + Transformer) ---
        m_ens = evaluate_model(y_te, (p_svm + p_tr) / 2.0)
        fold_records.append({'dataset': dataset_name, 'model': 'Ensemble_SVM_Trans', 'fold': fold+1,
                              'accuracy': m_ens['acc'], 'auc': m_ens['auc'], 'f1': m_ens['f1']})

        del svm, mlp

    return fold_records

print('CV runner defined ✅')

CV runner defined ✅


In [5]:
X_3d_166   = np.load(f'{FEATURES_DIR}/fc_tangent_3d.npy')
X_flat_166 = np.load(f'{FEATURES_DIR}/fc_tangent_flat.npy')
y          = np.load(f'{FEATURES_DIR}/labels_final.npy')

print(f'Subjects: {len(y)}  |  HC: {np.sum(y==0)}  |  ZAN: {np.sum(y==1)}')
print('\nRunning 10-Fold CV — Full Brain (166 ROIs)...')

records_166 = run_full_cv(X_flat_166, X_3d_166, y, num_rois=166, dataset_name='166_ROIs')
print('\n✅ 166 ROI fold-level results collected')

Subjects: 51  |  HC: 27  |  ZAN: 24

Running 10-Fold CV — Full Brain (166 ROIs)...

✅ 166 ROI fold-level results collected


In [6]:
X_3d_60   = np.load(f'{FEATURES_DIR}/fc_targeted_tangent_3d.npy')
X_flat_60 = np.load(f'{FEATURES_DIR}/fc_targeted_tangent_flat.npy')
y         = np.load(f'{FEATURES_DIR}/labels_final.npy')

print(f'Subjects: {len(y)}  |  HC: {np.sum(y==0)}  |  ZAN: {np.sum(y==1)}')
print('\nRunning 10-Fold CV — Targeted (60 ROIs)...')

records_60 = run_full_cv(X_flat_60, X_3d_60, y, num_rois=60, dataset_name='60_ROIs')
print('\n✅ 60 ROI fold-level results collected')

Subjects: 51  |  HC: 27  |  ZAN: 24

Running 10-Fold CV — Targeted (60 ROIs)...

✅ 60 ROI fold-level results collected


In [7]:
fold_df = pd.DataFrame(records_166 + records_60)
fold_df.to_csv(f'{FEATURES_DIR}/per_fold_results.csv', index=False)

print('Per-fold results (first 12 rows):')
print(fold_df.head(12).to_string(index=False))

print('\n\nSanity check — recomputed mean ± std (should match your original table):')
summary_check = (
    fold_df.groupby(['dataset','model'])[['accuracy','auc','f1']]
    .agg(['mean','std'])
)
print(summary_check)

print('\n✅ Saved: per_fold_results.csv  (one row per fold per model per dataset)')

Per-fold results (first 12 rows):
 dataset               model  fold  accuracy      auc       f1
166_ROIs             SVM-RBF     1  0.333333 0.222222 0.333333
166_ROIs                 MLP     1  0.333333 0.222222 0.333333
166_ROIs BrainNetTransformer     1  0.833333 1.000000 0.857143
166_ROIs  Ensemble_SVM_Trans     1  0.833333 1.000000 0.857143
166_ROIs             SVM-RBF     2  1.000000 1.000000 1.000000
166_ROIs                 MLP     2  1.000000 1.000000 1.000000
166_ROIs BrainNetTransformer     2  0.800000 1.000000 0.800000
166_ROIs  Ensemble_SVM_Trans     2  1.000000 1.000000 1.000000
166_ROIs             SVM-RBF     3  0.800000 0.666667 0.666667
166_ROIs                 MLP     3  0.600000 0.666667 0.500000
166_ROIs BrainNetTransformer     3  0.400000 0.333333 0.000000
166_ROIs  Ensemble_SVM_Trans     3  0.800000 0.666667 0.666667


Sanity check — recomputed mean ± std (should match your original table):
                              accuracy                 auc            \


In [8]:
print(fold_df.shape)                          # expect (80, 5)
print(fold_df.groupby('dataset')['fold'].nunique())   # expect 10 for both datasets
print(fold_df[fold_df['dataset']=='166_ROIs']['fold'].tolist())  # expect 1-10, each appearing 4x

(80, 6)
dataset
166_ROIs    10
60_ROIs     10
Name: fold, dtype: int64
[1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8, 9, 9, 9, 9, 10, 10, 10, 10]


In [9]:
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

for dataset_name in fold_df['dataset'].unique():
    print(f'\n{"="*70}')
    print(f'  {dataset_name} — All 10 Folds')
    print(f'{"="*70}')
    subset = fold_df[fold_df['dataset'] == dataset_name].copy()
    subset = subset.sort_values(['fold', 'model'])
    print(subset[['fold', 'model', 'accuracy', 'auc', 'f1']].to_string(index=False))


  166_ROIs — All 10 Folds
 fold               model  accuracy      auc       f1
    1 BrainNetTransformer  0.833333 1.000000 0.857143
    1  Ensemble_SVM_Trans  0.833333 1.000000 0.857143
    1                 MLP  0.333333 0.222222 0.333333
    1             SVM-RBF  0.333333 0.222222 0.333333
    2 BrainNetTransformer  0.800000 1.000000 0.800000
    2  Ensemble_SVM_Trans  1.000000 1.000000 1.000000
    2                 MLP  1.000000 1.000000 1.000000
    2             SVM-RBF  1.000000 1.000000 1.000000
    3 BrainNetTransformer  0.400000 0.333333 0.000000
    3  Ensemble_SVM_Trans  0.800000 0.666667 0.666667
    3                 MLP  0.600000 0.666667 0.500000
    3             SVM-RBF  0.800000 0.666667 0.666667
    4 BrainNetTransformer  0.400000 0.666667 0.000000
    4  Ensemble_SVM_Trans  0.600000 0.500000 0.500000
    4                 MLP  0.600000 0.666667 0.666667
    4             SVM-RBF  0.400000 0.500000 0.400000
    5 BrainNetTransformer  0.600000 0.500000 0.666667
 

In [10]:
for metric in ['accuracy', 'auc', 'f1']:
    print(f'\n{"="*70}')
    print(f'  {metric.upper()} — per fold, per model')
    print(f'{"="*70}')
    for dataset_name in fold_df['dataset'].unique():
        print(f'\n--- {dataset_name} ---')
        pivot = fold_df[fold_df['dataset']==dataset_name].pivot(
            index='fold', columns='model', values=metric
        )
        print(pivot.round(3).to_string())


  ACCURACY — per fold, per model

--- 166_ROIs ---
model  BrainNetTransformer  Ensemble_SVM_Trans    MLP  SVM-RBF
fold                                                          
1                    0.833               0.833  0.333    0.333
2                    0.800               1.000  1.000    1.000
3                    0.400               0.800  0.600    0.800
4                    0.400               0.600  0.600    0.400
5                    0.600               0.800  0.800    0.800
6                    0.000               0.400  0.400    0.600
7                    0.800               0.800  0.600    0.600
8                    0.600               0.600  0.800    0.600
9                    0.600               0.400  0.200    0.400
10                   0.800               0.800  0.400    0.600

--- 60_ROIs ---
model  BrainNetTransformer  Ensemble_SVM_Trans  MLP  SVM-RBF
fold                                                        
1                    0.833                 0.5  0.5  

In [11]:
from sklearn.linear_model import LogisticRegression

def run_full_cv_with_lasso(X_flat, X_3d, y, num_rois, dataset_name, n_splits=10, seed=42, lasso_k=30):
    """
    Same as run_full_cv, but adds LASSO (Top-30 features) as a 5th model
    for direct comparison under identical 10-fold conditions.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_records = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_flat, y)):
        print(f'  [{dataset_name}] Fold {fold+1}/{n_splits}', end='\r')
        y_tr, y_te = y[tr_idx], y[te_idx]

        # --- shared preprocessing (Top 300, for SVM/MLP/Ensemble) ---
        scaler  = StandardScaler()
        Xtr_sc  = scaler.fit_transform(X_flat[tr_idx])
        Xte_sc  = scaler.transform(X_flat[te_idx])

        sel     = SelectKBest(f_classif, k=300)
        Xtr_sel = sel.fit_transform(Xtr_sc, y_tr)
        Xte_sel = sel.transform(Xte_sc)

        # --- SVM ---
        svm = SVC(kernel='rbf', probability=True, random_state=seed)
        svm.fit(Xtr_sel, y_tr)
        p_svm = svm.predict_proba(Xte_sel)[:, 1]
        m_svm = evaluate_model(y_te, p_svm)
        fold_records.append({'dataset': dataset_name, 'model': 'SVM-RBF', 'fold': fold+1,
                              'accuracy': m_svm['acc'], 'auc': m_svm['auc'], 'f1': m_svm['f1']})

        # --- MLP ---
        mlp = MLPClassifier(hidden_layer_sizes=(128,64), alpha=0.1, max_iter=500, random_state=seed)
        mlp.fit(Xtr_sel, y_tr)
        p_mlp = mlp.predict_proba(Xte_sel)[:, 1]
        m_mlp = evaluate_model(y_te, p_mlp)
        fold_records.append({'dataset': dataset_name, 'model': 'MLP', 'fold': fold+1,
                              'accuracy': m_mlp['acc'], 'auc': m_mlp['auc'], 'f1': m_mlp['f1']})

        # --- BrainNetTransformer ---
        p_tr = run_transformer_fold(X_3d[tr_idx], y_tr, X_3d[te_idx], y_te, num_rois=num_rois)
        m_tr = evaluate_model(y_te, p_tr)
        fold_records.append({'dataset': dataset_name, 'model': 'BrainNetTransformer', 'fold': fold+1,
                              'accuracy': m_tr['acc'], 'auc': m_tr['auc'], 'f1': m_tr['f1']})

        # --- Ensemble (SVM + Transformer) ---
        m_ens = evaluate_model(y_te, (p_svm + p_tr) / 2.0)
        fold_records.append({'dataset': dataset_name, 'model': 'Ensemble_SVM_Trans', 'fold': fold+1,
                              'accuracy': m_ens['acc'], 'auc': m_ens['auc'], 'f1': m_ens['f1']})

        # --- LASSO (Top 30, your best model — separate preprocessing to match your original pipeline) ---
        sel_lasso     = SelectKBest(f_classif, k=lasso_k)
        Xtr_lasso     = sel_lasso.fit_transform(Xtr_sc, y_tr)
        Xte_lasso     = sel_lasso.transform(Xte_sc)
        lasso         = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=seed)
        lasso.fit(Xtr_lasso, y_tr)
        p_lasso       = lasso.predict_proba(Xte_lasso)[:, 1]
        m_lasso       = evaluate_model(y_te, p_lasso)
        fold_records.append({'dataset': dataset_name, 'model': 'LASSO_Top30', 'fold': fold+1,
                              'accuracy': m_lasso['acc'], 'auc': m_lasso['auc'], 'f1': m_lasso['f1']})

        del svm, mlp, lasso

    return fold_records

In [12]:
records_166 = run_full_cv_with_lasso(X_flat_166, X_3d_166, y, num_rois=166, dataset_name='166_ROIs')
records_60  = run_full_cv_with_lasso(X_flat_60,  X_3d_60,  y, num_rois=60,  dataset_name='60_ROIs')

fold_df = pd.DataFrame(records_166 + records_60)
fold_df.to_csv(f'{FEATURES_DIR}/per_fold_results_with_lasso.csv', index=False)

In [13]:
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

# 1. Aggregated mean ± std per model per dataset (this is your main results table for the paper)
print('='*70)
print('  SUMMARY: Mean ± Std across 10 folds')
print('='*70)
summary = fold_df.groupby(['dataset', 'model'])[['accuracy', 'auc', 'f1']].agg(['mean', 'std'])
print(summary.round(4))

# 2. Full per-fold breakdown, one table per dataset
for dataset_name in fold_df['dataset'].unique():
    print(f'\n{"="*70}')
    print(f'  {dataset_name} — All 10 Folds')
    print(f'{"="*70}')
    subset = fold_df[fold_df['dataset'] == dataset_name].sort_values(['fold', 'model'])
    print(subset[['fold', 'model', 'accuracy', 'auc', 'f1']].to_string(index=False))

# 3. Pivoted view — fold as rows, model as columns (good for supplementary table)
for metric in ['accuracy', 'auc', 'f1']:
    print(f'\n{"="*70}')
    print(f'  {metric.upper()} — per fold, per model')
    print(f'{"="*70}')
    for dataset_name in fold_df['dataset'].unique():
        print(f'\n--- {dataset_name} ---')
        pivot = fold_df[fold_df['dataset']==dataset_name].pivot(index='fold', columns='model', values=metric)
        print(pivot.round(3).to_string())

# 4. Sanity check — shape and fold count
print(f'\n\nTotal rows: {fold_df.shape[0]}  (expect 100: 2 datasets × 10 folds × 5 models)')
print(fold_df.groupby('dataset')['fold'].nunique())

  SUMMARY: Mean ± Std across 10 folds
                             accuracy             auc              f1        
                                 mean     std    mean     std    mean     std
dataset  model                                                               
166_ROIs BrainNetTransformer   0.6267  0.1481  0.5444  0.2108  0.5033  0.2945
         Ensemble_SVM_Trans    0.6667  0.1633  0.6222  0.2339  0.5340  0.3323
         LASSO_Top30           0.5733  0.1838  0.7056  0.2328  0.5295  0.2488
         MLP                   0.5733  0.2459  0.6056  0.2651  0.5329  0.2873
         SVM-RBF               0.6133  0.2080  0.5556  0.2645  0.5133  0.2644
60_ROIs  BrainNetTransformer   0.5467  0.1398  0.4778  0.3023  0.5176  0.2141
         Ensemble_SVM_Trans    0.6100  0.2767  0.6056  0.3853  0.5629  0.3212
         LASSO_Top30           0.7433  0.2331  0.8111  0.1998  0.6890  0.3207
         MLP                   0.5700  0.2003  0.6111  0.2759  0.5419  0.2633
         SVM-RBF          

In [18]:
!pip install nilearn
import numpy as np
import pandas as pd
from nilearn.connectome import ConnectivityMeasure

# Install nilearn if not already installed

final_df = pd.read_csv(f'{FEATURES_DIR}/valid_subjects_metadata.csv')

# Load all 166-ROI timeseries (same subjects used for your targeted 60-ROI set)
all_ts_166 = []
valid_indices = []

for idx, row in final_df.iterrows():
    ts_path = f'{FEATURES_DIR}/{row["participant_id"]}_timeseries.npy'
    if os.path.exists(ts_path):
        ts = np.load(ts_path)
        if ts.shape[1] == 166:
            all_ts_166.append(ts)
            valid_indices.append(idx)

clean_df = final_df.iloc[valid_indices].reset_index(drop=True)
print(f'Loaded {len(all_ts_166)} subjects with 166-ROI timeseries')

# Pick 60 RANDOM ROIs (same size as your targeted set, but no biological rationale)
np.random.seed(123)  # different seed from model training — purely for reproducible ROI sampling
random_roi_idx = sorted(np.random.choice(166, size=60, replace=False))
print(f'Selected random ROI indices (first 10): {random_roi_idx[:10]}...')

# Extract random-ROI timeseries per subject
all_ts_random60 = [ts[:, random_roi_idx] for ts in all_ts_166]

# Compute tangent-space FC on the random 60 ROIs — identical method to your targeted set
conn_measure_rand = ConnectivityMeasure(kind='tangent', vectorize=False)
fc_random60 = conn_measure_rand.fit_transform(all_ts_random60)

triu_idx_rand = np.triu_indices(60, k=1)
X_flat_random60 = np.array([fc[triu_idx_rand] for fc in fc_random60])

y_random60 = clean_df['label'].values

np.save(f'{FEATURES_DIR}/fc_random60_tangent_flat.npy', X_flat_random60)
np.save(f'{FEATURES_DIR}/labels_random60.npy', y_random60)

print(f'Random-60-ROI flat features shape: {X_flat_random60.shape}')
print(f'Subjects: {len(y_random60)}  |  HC: {np.sum(y_random60==0)}  |  ZAN: {np.sum(y_random60==1)}')


Loaded 51 subjects with 166-ROI timeseries
Selected random ROI indices (first 10): [np.int64(0), np.int64(4), np.int64(5), np.int64(7), np.int64(8), np.int64(13), np.int64(16), np.int64(19), np.int64(20), np.int64(23)]...
Random-60-ROI flat features shape: (51, 1770)
Subjects: 51  |  HC: 27  |  ZAN: 24


In [19]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

def run_lasso_only_cv(X_flat, y, dataset_name, n_splits=10, seed=42, lasso_k=30):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_records = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_flat, y)):
        y_tr, y_te = y[tr_idx], y[te_idx]

        scaler  = StandardScaler()
        Xtr_sc  = scaler.fit_transform(X_flat[tr_idx])
        Xte_sc  = scaler.transform(X_flat[te_idx])

        sel     = SelectKBest(f_classif, k=lasso_k)
        Xtr_sel = sel.fit_transform(Xtr_sc, y_tr)
        Xte_sel = sel.transform(Xte_sc)

        lasso = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=seed)
        lasso.fit(Xtr_sel, y_tr)
        p_lasso = lasso.predict_proba(Xte_sel)[:, 1]
        m = evaluate_model(y_te, p_lasso)

        fold_records.append({'dataset': dataset_name, 'model': 'LASSO_Top30', 'fold': fold+1,
                              'accuracy': m['acc'], 'auc': m['auc'], 'f1': m['f1']})
        print(f'  [{dataset_name}] Fold {fold+1}/{n_splits}: Acc={m["acc"]:.3f}, AUC={m["auc"]:.3f}')

    return fold_records

# Run on random 60 ROIs
records_random60 = run_lasso_only_cv(X_flat_random60, y_random60, dataset_name='60_ROIs_RANDOM')

pd.DataFrame(records_random60).to_csv(f'{FEATURES_DIR}/lasso_random60_results.csv', index=False)
print('✅ Random-ROI control results saved')

  [60_ROIs_RANDOM] Fold 1/10: Acc=0.000, AUC=0.000
  [60_ROIs_RANDOM] Fold 2/10: Acc=0.800, AUC=0.667
  [60_ROIs_RANDOM] Fold 3/10: Acc=0.400, AUC=0.500
  [60_ROIs_RANDOM] Fold 4/10: Acc=0.800, AUC=0.500
  [60_ROIs_RANDOM] Fold 5/10: Acc=0.600, AUC=0.833
  [60_ROIs_RANDOM] Fold 6/10: Acc=0.400, AUC=0.333
  [60_ROIs_RANDOM] Fold 7/10: Acc=0.400, AUC=0.667
  [60_ROIs_RANDOM] Fold 8/10: Acc=0.600, AUC=0.833
  [60_ROIs_RANDOM] Fold 9/10: Acc=0.600, AUC=0.667
  [60_ROIs_RANDOM] Fold 10/10: Acc=0.600, AUC=0.500
✅ Random-ROI control results saved


In [20]:
# Your existing targeted-60-ROI LASSO results (from fold_df, already computed)
targeted_lasso = fold_df[(fold_df['dataset']=='60_ROIs') & (fold_df['model']=='LASSO_Top30')].copy()
targeted_lasso['dataset'] = '60_ROIs_TARGETED'

random_lasso = pd.DataFrame(records_random60)

comparison = pd.concat([targeted_lasso, random_lasso], ignore_index=True)

print('='*70)
print('  NEGATIVE CONTROL: Targeted (biological) 60 ROIs vs Random 60 ROIs')
print('='*70)
summary_compare = comparison.groupby('dataset')[['accuracy','auc','f1']].agg(['mean','std'])
print(summary_compare.round(4))

# Per-fold side-by-side view
print('\nPer-fold comparison:')
pivot_compare = comparison.pivot(index='fold', columns='dataset', values='accuracy')
print(pivot_compare.round(3).to_string())

  NEGATIVE CONTROL: Targeted (biological) 60 ROIs vs Random 60 ROIs
                 accuracy             auc             f1        
                     mean     std    mean     std   mean     std
dataset                                                         
60_ROIs_RANDOM     0.5200  0.2348  0.5500  0.2491  0.485  0.2894
60_ROIs_TARGETED   0.7433  0.2331  0.8111  0.1998  0.689  0.3207

Per-fold comparison:
dataset  60_ROIs_RANDOM  60_ROIs_TARGETED
fold                                     
1                   0.0             0.833
2                   0.8             1.000
3                   0.4             0.400
4                   0.8             0.800
5                   0.6             0.400
6                   0.4             0.600
7                   0.4             0.800
8                   0.6             1.000
9                   0.6             0.600
10                  0.6             1.000


In [21]:
from scipy.stats import ttest_rel, wilcoxon

# --- Paired comparison (same fold structure, same seed=42, same n_splits=10) ---
targeted_acc = comparison[comparison['dataset']=='60_ROIs_TARGETED'].sort_values('fold')['accuracy'].values
random_acc   = comparison[comparison['dataset']=='60_ROIs_RANDOM'].sort_values('fold')['accuracy'].values

targeted_auc = comparison[comparison['dataset']=='60_ROIs_TARGETED'].sort_values('fold')['auc'].values
random_auc   = comparison[comparison['dataset']=='60_ROIs_RANDOM'].sort_values('fold')['auc'].values

print('='*70)
print('  STATISTICAL TEST: Targeted ROIs vs Random ROIs (paired, per-fold)')
print('='*70)

# Paired t-test
t_stat_acc, p_acc = ttest_rel(targeted_acc, random_acc)
t_stat_auc, p_auc = ttest_rel(targeted_auc, random_auc)

print(f'\nAccuracy — Paired t-test: t={t_stat_acc:.3f}, p={p_acc:.4f}  '
      f'({"SIGNIFICANT ✅" if p_acc < 0.05 else "NOT SIGNIFICANT ❌"})')
print(f'AUC      — Paired t-test: t={t_stat_auc:.3f}, p={p_auc:.4f}  '
      f'({"SIGNIFICANT ✅" if p_auc < 0.05 else "NOT SIGNIFICANT ❌"})')

# Wilcoxon signed-rank (non-parametric, more robust for n=10 folds)
try:
    w_stat_acc, wp_acc = wilcoxon(targeted_acc, random_acc)
    w_stat_auc, wp_auc = wilcoxon(targeted_auc, random_auc)
    print(f'\nAccuracy — Wilcoxon signed-rank: p={wp_acc:.4f}  '
          f'({"SIGNIFICANT ✅" if wp_acc < 0.05 else "NOT SIGNIFICANT ❌"})')
    print(f'AUC      — Wilcoxon signed-rank: p={wp_auc:.4f}  '
          f'({"SIGNIFICANT ✅" if wp_auc < 0.05 else "NOT SIGNIFICANT ❌"})')
except ValueError as e:
    print(f'\nWilcoxon test skipped: {e}')

# Effect size (Cohen's d for paired samples)
diff_acc = targeted_acc - random_acc
cohens_d_acc = np.mean(diff_acc) / np.std(diff_acc, ddof=1)
print(f'\nMean accuracy difference (targeted - random): {np.mean(diff_acc):.4f}')
print(f'Cohen\'s d (paired): {cohens_d_acc:.3f}')

# --- Save everything ---
comparison.to_csv(f'{FEATURES_DIR}/negative_control_comparison.csv', index=False)

stats_summary = pd.DataFrame([
    {'Metric': 'Accuracy', 'Targeted_Mean': targeted_acc.mean(), 'Random_Mean': random_acc.mean(),
     'Diff': targeted_acc.mean() - random_acc.mean(), 'p_ttest': p_acc, 'Cohens_d': cohens_d_acc},
    {'Metric': 'AUC', 'Targeted_Mean': targeted_auc.mean(), 'Random_Mean': random_auc.mean(),
     'Diff': targeted_auc.mean() - random_auc.mean(), 'p_ttest': p_auc, 'Cohens_d': None},
])
stats_summary.to_csv(f'{FEATURES_DIR}/negative_control_stats.csv', index=False)

print('\n✅ Saved: negative_control_comparison.csv, negative_control_stats.csv')

  STATISTICAL TEST: Targeted ROIs vs Random ROIs (paired, per-fold)

Accuracy — Paired t-test: t=2.375, p=0.0415  (SIGNIFICANT ✅)
AUC      — Paired t-test: t=2.469, p=0.0356  (SIGNIFICANT ✅)

Accuracy — Wilcoxon signed-rank: p=0.0625  (NOT SIGNIFICANT ❌)
AUC      — Wilcoxon signed-rank: p=0.0625  (NOT SIGNIFICANT ❌)

Mean accuracy difference (targeted - random): 0.2233
Cohen's d (paired): 0.751

✅ Saved: negative_control_comparison.csv, negative_control_stats.csv


In [22]:
print('Labels match between targeted and random runs:', np.array_equal(y, y_random60))

Labels match between targeted and random runs: True


In [23]:
from scipy.stats import ttest_rel, wilcoxon

targeted_acc = comparison[comparison['dataset']=='60_ROIs_TARGETED'].sort_values('fold')['accuracy'].values
random_acc   = comparison[comparison['dataset']=='60_ROIs_RANDOM'].sort_values('fold')['accuracy'].values
targeted_auc = comparison[comparison['dataset']=='60_ROIs_TARGETED'].sort_values('fold')['auc'].values
random_auc   = comparison[comparison['dataset']=='60_ROIs_RANDOM'].sort_values('fold')['auc'].values

t_stat_acc, p_acc = ttest_rel(targeted_acc, random_acc)
t_stat_auc, p_auc = ttest_rel(targeted_auc, random_auc)

print(f'Accuracy — Paired t-test: t={t_stat_acc:.3f}, p={p_acc:.4f}')
print(f'AUC      — Paired t-test: t={t_stat_auc:.3f}, p={p_auc:.4f}')

diff_acc = targeted_acc - random_acc
cohens_d = np.mean(diff_acc) / np.std(diff_acc, ddof=1)
print(f'Cohen\'s d (accuracy): {cohens_d:.3f}')

comparison.to_csv(f'{FEATURES_DIR}/negative_control_comparison.csv', index=False)

Accuracy — Paired t-test: t=2.375, p=0.0415
AUC      — Paired t-test: t=2.469, p=0.0356
Cohen's d (accuracy): 0.751
